# 🖼️ BLIP-2 Caption Server (Fine-tuned on COCO)

**Run this on Google Colab with a T4 GPU (free).**
It loads your fine-tuned BLIP-2 OPT-6.7B + LoRA and creates a public Gradio URL.
Your local Streamlit app connects to this URL.

In [ ]:
!pip install -q transformers accelerate peft gradio pillow torch

In [ ]:
import torch
from transformers import Blip2ForConditionalGeneration, Blip2Processor
from peft import PeftModel
from PIL import Image
import gradio as gr

ADAPTER_REPO = "Parthg0106/DL-mini"
BASE_MODEL = "Salesforce/blip2-opt-6.7b"

print("Loading processor...")
processor = Blip2Processor.from_pretrained(ADAPTER_REPO)

print(f"Loading {BASE_MODEL}...")
model = Blip2ForConditionalGeneration.from_pretrained(
    BASE_MODEL, torch_dtype=torch.float16, device_map="auto"
)

print("Applying LoRA weights...")
model = PeftModel.from_pretrained(model, ADAPTER_REPO)
model.eval()

print(f"✅ Model loaded on {model.device}")
print(f"VRAM: {torch.cuda.memory_allocated()/1024**3:.1f} GB")

In [ ]:
def caption(image, num_beams=5, max_tokens=50, temperature=0.7):
    if image is None:
        return "Upload an image."
    
    # Handle different input formats
    if isinstance(image, dict):
        image = Image.open(image["path"]).convert("RGB")
    elif isinstance(image, str):
        image = Image.open(image).convert("RGB")
    elif not isinstance(image, Image.Image):
        image = Image.fromarray(image).convert("RGB")
    else:
        image = image.convert("RGB")
    
    inputs = processor(images=image, return_tensors="pt")
    pv = inputs["pixel_values"].to(model.device, dtype=torch.float16)
    
    with torch.no_grad():
        out = model.generate(
            pixel_values=pv,
            max_new_tokens=int(max_tokens),
            num_beams=int(num_beams),
            do_sample=temperature > 0.1,
            temperature=max(float(temperature), 0.1),
            early_stopping=True,
        )
    return processor.batch_decode(out, skip_special_tokens=True)[0].strip()

demo = gr.Interface(
    fn=caption,
    inputs=[
        gr.Image(label="Upload Image"),
        gr.Slider(1, 10, value=5, step=1, label="Beam Search Width"),
        gr.Slider(25, 100, value=50, step=5, label="Max Tokens"),
        gr.Slider(0.1, 1.5, value=0.7, step=0.1, label="Temperature"),
    ],
    outputs=gr.Textbox(label="Caption"),
    title="🖼️ BLIP-2 Image Captioner (Fine-tuned)",
    description="BLIP-2 OPT-6.7B + LoRA fine-tuned on MS-COCO (BLEU-4: 0.42)",
)

demo.launch(share=True, debug=True, show_error=True)


## 📋 How to use

1. Run all cells above
2. Copy the **public URL** (e.g. `https://xxxxx.gradio.live`)
3. On your local PC, set the environment variable:
   ```
   set GRADIO_API_URL=https://xxxxx.gradio.live
   ```
4. Run your local Streamlit app: `streamlit run app.py`

The Gradio URL is valid for **72 hours**. Keep this Colab tab open.